<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/04_construction_review_indicators_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การตรวจรูปแบบโครงการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

ผลลัพธ์ใช้สำหรับจัดลำดับการตรวจเอกสาร ไม่ใช่หลักฐานว่ามีการทุจริตหรือแบ่งซื้อแบ่งจ้าง

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
MUTED = '#667085'
GRID = '#E4E7EC'

In [ ]:
processed_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/processed'
)

figure_dir = processed_dir.parents[3] / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

project_path = processed_dir / 'construction_projects_2569.csv'
contract_path = processed_dir / 'construction_contracts_2569.csv'

project_data = pd.read_csv(project_path, low_memory=False)
contract_data = pd.read_csv(contract_path, low_memory=False)

print(f'Project rows: {len(project_data):,}')
print(f'Contract–supplier rows: {len(contract_data):,}')
print(f'Figure directory: {figure_dir}')

In [ ]:
project_id_column = 'รหัสโครงการ'
project_name_column = 'ชื่อโครงการจัดซื้อจัดจ้าง'
agency_column = 'ชื่อหน่วยงาน'
subagency_column = 'ชื่อหน่วยงานย่อย'
province_column = 'จังหวัด'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'
budget_column = 'วงเงินงบประมาณ (บาท)'
reference_price_column = 'ราคากลาง (บาท)'
awarded_price_column = 'ราคาที่ตกลงซื้อ / จ้าง ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
transaction_date_column = 'วันที่เกิดรายการ'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'

specific_method_value = 'เฉพาะเจาะจง'
legal_budget_ceiling = 500_000

project_data['in_study_scope'] = (
    project_data[method_column].eq(specific_method_value)
    & project_data[budget_column].le(legal_budget_ceiling)
)

study_project_data = project_data.loc[
    project_data['in_study_scope']
].copy()

assert len(project_data) == 178_978
assert len(study_project_data) == 136_070

study_project_ids = set(study_project_data[project_id_column])

study_summary = pd.Series({
    'โครงการก่อสร้างทั้งหมด': len(project_data),
    'โครงการในกลุ่มศึกษา': len(study_project_data),
    'สัดส่วนโครงการทั้งหมด (%)': (
        len(study_project_data) / len(project_data) * 100
    )
}, name='value')

display(study_summary.to_frame())

## 1. เตรียมข้อมูลผู้รับจ้าง

In [ ]:
contract_data['is_joint_venture_member'] = (
    contract_data[supplier_name_column]
    .astype('string')
    .str.contains('สัญญากิจการค้าร่วม', na=False)
)

supplier_entity_data = contract_data.loc[
    ~contract_data['is_joint_venture_member']
    & contract_data[supplier_id_column].notna()
].copy()

project_supplier_data = (
    supplier_entity_data
    .groupby(
        [project_id_column, supplier_id_column],
        dropna=False
    )
    .agg(
        supplier_name=(supplier_name_column, 'first'),
        supplier_awarded_value=(contract_value_column, 'sum')
    )
    .reset_index()
    .merge(
        project_data[
            [
                project_id_column,
                project_name_column,
                agency_column,
                subagency_column,
                province_column,
                method_column,
                budget_column,
                awarded_price_column,
                transaction_date_column,
                'in_study_scope'
            ]
        ],
        on=project_id_column,
        how='left',
        validate='many_to_one'
    )
)

print(f'แถวสมาชิก Joint Venture ที่ไม่นับซ้ำ: {contract_data["is_joint_venture_member"].sum():,}')
print(f'คู่โครงการ–ผู้รับจ้าง: {len(project_supplier_data):,}')
print(f'โครงการที่มีรหัสผู้รับจ้าง: {project_supplier_data[project_id_column].nunique():,}')

In [ ]:
supplier_value_by_project = (
    project_supplier_data
    .groupby(project_id_column)['supplier_awarded_value']
    .sum()
    .rename('supplier_value_sum')
)

supplier_reconciliation = (
    project_data[
        [project_id_column, awarded_price_column]
    ]
    .merge(
        supplier_value_by_project,
        on=project_id_column,
        how='left'
    )
)

supplier_reconciliation['is_matched'] = np.isclose(
    supplier_reconciliation['supplier_value_sum'],
    supplier_reconciliation[awarded_price_column],
    rtol=0,
    atol=1,
    equal_nan=False
)

reconciliation_summary = pd.Series({
    'โครงการที่ตรวจ': len(supplier_reconciliation),
    'โครงการที่มูลค่าตรงกัน': supplier_reconciliation['is_matched'].sum(),
    'โครงการที่มูลค่าไม่ตรงกัน': (~supplier_reconciliation['is_matched']).sum(),
    'สัดส่วนที่ตรงกัน (%)': supplier_reconciliation['is_matched'].mean() * 100
}, name='value')

display(reconciliation_summary.to_frame())

### ผลการเตรียมข้อมูล

แถวสมาชิก Joint Venture ไม่นำมารวมซ้ำในการคำนวณมูลค่าของผู้รับจ้าง

## 2. Pattern 1 — โครงการใกล้เพดานที่เกิดซ้ำ

เงื่อนไขหลัก:

- วิธีเฉพาะเจาะจง
- วงเงิน 490,000–500,000 บาท
- หน่วยงานย่อย ผู้รับจ้าง และวันที่เกิดรายการเดียวกัน
- อย่างน้อย 3 โครงการ

In [ ]:
near_ceiling_windows = [
    (450_000, '450,000–500,000'),
    (480_000, '480,000–500,000'),
    (490_000, '490,000–500,000'),
    (495_000, '495,000–500,000')
]

window_records = []

for lower_bound, label in near_ceiling_windows:
    valid_cluster_rows = (
        project_supplier_data[subagency_column].notna()
        & project_supplier_data[supplier_id_column].notna()
        & project_supplier_data[transaction_date_column].notna()
        & project_supplier_data[transaction_date_column].ne('-')
    )

    window_data = project_supplier_data.loc[
        project_supplier_data['in_study_scope']
        & valid_cluster_rows
        & project_supplier_data[budget_column].between(
            lower_bound,
            legal_budget_ceiling,
            inclusive='both'
        )
    ]

    window_clusters = (
        window_data
        .groupby(
            [
                subagency_column,
                supplier_id_column,
                transaction_date_column
            ],
            dropna=False
        )[project_id_column]
        .nunique()
    )

    selected_clusters = window_clusters.loc[
        window_clusters.ge(3)
    ]

    window_records.append({
        'ช่วงวงเงิน': label,
        'โครงการในช่วง': window_data[project_id_column].nunique(),
        'กลุ่มอย่างน้อย 3 โครงการ': len(selected_clusters),
        'โครงการในกลุ่ม': selected_clusters.sum()
    })

near_ceiling_sensitivity = pd.DataFrame(window_records)
display(near_ceiling_sensitivity)

In [ ]:
valid_cluster_rows = (
    project_supplier_data[subagency_column].notna()
    & project_supplier_data[supplier_id_column].notna()
    & project_supplier_data[transaction_date_column].notna()
    & project_supplier_data[transaction_date_column].ne('-')
)

near_ceiling_data = project_supplier_data.loc[
    project_supplier_data['in_study_scope']
    & valid_cluster_rows
    & project_supplier_data[budget_column].between(
        490_000,
        legal_budget_ceiling,
        inclusive='both'
    )
].copy()

cluster_columns = [
    subagency_column,
    supplier_id_column,
    transaction_date_column
]

near_ceiling_clusters = (
    near_ceiling_data
    .groupby(cluster_columns, dropna=False)
    .agg(
        supplier_name=('supplier_name', 'first'),
        project_count=(project_id_column, 'nunique'),
        total_budget=(budget_column, 'sum')
    )
    .reset_index()
)

repeated_clusters = (
    near_ceiling_clusters.loc[
        near_ceiling_clusters['project_count'].ge(3)
    ]
    .sort_values(
        ['project_count', 'total_budget'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern1_project_ids = set(
    near_ceiling_data
    .merge(
        repeated_clusters[cluster_columns],
        on=cluster_columns,
        how='inner',
        validate='many_to_many'
    )[project_id_column]
)

project_data['flag_pattern_1'] = (
    project_data[project_id_column].isin(pattern1_project_ids)
)

pattern1_summary = pd.Series({
    'กลุ่มที่ผ่านเกณฑ์': len(repeated_clusters),
    'โครงการที่เกี่ยวข้อง': project_data['flag_pattern_1'].sum(),
    'สัดส่วนกลุ่มศึกษา (%)': (
        project_data['flag_pattern_1'].sum()
        / len(study_project_data)
        * 100
    )
}, name='value')

display(pattern1_summary.to_frame())
display(repeated_clusters.head(15))

In [ ]:
cluster_size_options = [2, 3, 5, 10]

cluster_size_sensitivity = pd.DataFrame([
    {
        'จำนวนขั้นต่ำต่อกลุ่ม': minimum_size,
        'จำนวนกลุ่ม': near_ceiling_clusters['project_count'].ge(minimum_size).sum(),
        'จำนวนโครงการ': near_ceiling_clusters.loc[
            near_ceiling_clusters['project_count'].ge(minimum_size),
            'project_count'
        ].sum()
    }
    for minimum_size in cluster_size_options
])

display(cluster_size_sensitivity)

### ผล Pattern 1

พบ 1,245 กลุ่ม รวม 5,623 โครงการตามเกณฑ์หลัก

## 3. Pattern 2 — การพึ่งพาผู้รับจ้างภายในหน่วยงาน

เงื่อนไขหลัก:

- ผู้รับจ้างได้รับอย่างน้อย 10 โครงการ
- ครองอย่างน้อย 75% ของจำนวนโครงการในหน่วยงานย่อย
- ครองอย่างน้อย 75% ของมูลค่าในหน่วยงานย่อย

In [ ]:
study_supplier_data = project_supplier_data.loc[
    project_supplier_data['in_study_scope']
    & project_supplier_data[subagency_column].notna()
    & project_supplier_data[supplier_id_column].notna()
].copy()

supplier_relationships = (
    study_supplier_data
    .groupby(
        [subagency_column, supplier_id_column],
        dropna=False
    )
    .agg(
        supplier_name=('supplier_name', 'first'),
        supplier_project_count=(project_id_column, 'nunique'),
        supplier_awarded_value=('supplier_awarded_value', 'sum')
    )
    .reset_index()
)

subagency_totals = (
    study_project_data
    .groupby(subagency_column, dropna=False)
    .agg(
        subagency_project_count=(project_id_column, 'nunique'),
        subagency_awarded_value=(awarded_price_column, 'sum')
    )
    .reset_index()
)

supplier_relationships = supplier_relationships.merge(
    subagency_totals,
    on=subagency_column,
    how='left',
    validate='many_to_one'
)

supplier_relationships['project_share_pct'] = (
    supplier_relationships['supplier_project_count']
    / supplier_relationships['subagency_project_count']
    * 100
)

supplier_relationships['value_share_pct'] = (
    supplier_relationships['supplier_awarded_value']
    / supplier_relationships['subagency_awarded_value']
    * 100
)

display(
    supplier_relationships
    .sort_values('supplier_project_count', ascending=False)
    .head(15)
)

In [ ]:
minimum_project_options = [5, 10, 20]
share_threshold_options = [50, 75, 90]

dependence_records = []

for minimum_projects in minimum_project_options:
    for share_threshold in share_threshold_options:
        selected = supplier_relationships.loc[
            supplier_relationships['supplier_project_count'].ge(minimum_projects)
            & supplier_relationships['project_share_pct'].ge(share_threshold)
            & supplier_relationships['value_share_pct'].ge(share_threshold)
        ]

        dependence_records.append({
            'จำนวนโครงการขั้นต่ำ': minimum_projects,
            'ส่วนแบ่งขั้นต่ำ (%)': share_threshold,
            'จำนวนคู่หน่วยงาน–ผู้รับจ้าง': len(selected),
            'จำนวนหน่วยงานย่อย': selected[subagency_column].nunique(),
            'จำนวนผู้รับจ้าง': selected[supplier_id_column].nunique()
        })

supplier_dependence_sensitivity = pd.DataFrame(dependence_records)
display(supplier_dependence_sensitivity)

In [ ]:
high_dependence_relationships = (
    supplier_relationships.loc[
        supplier_relationships['supplier_project_count'].ge(10)
        & supplier_relationships['project_share_pct'].ge(75)
        & supplier_relationships['value_share_pct'].ge(75)
    ]
    .sort_values(
        ['project_share_pct', 'supplier_project_count'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern2_project_ids = set(
    study_supplier_data
    .merge(
        high_dependence_relationships[
            [subagency_column, supplier_id_column]
        ],
        on=[subagency_column, supplier_id_column],
        how='inner',
        validate='many_to_many'
    )[project_id_column]
)

project_data['flag_pattern_2'] = (
    project_data[project_id_column].isin(pattern2_project_ids)
)

pattern2_summary = pd.Series({
    'คู่หน่วยงาน–ผู้รับจ้างที่ผ่านเกณฑ์': len(high_dependence_relationships),
    'ผู้รับจ้างที่เกี่ยวข้อง': (
        high_dependence_relationships[supplier_id_column].nunique()
    ),
    'โครงการที่เกี่ยวข้อง': project_data['flag_pattern_2'].sum()
}, name='value')

display(pattern2_summary.to_frame())
display(high_dependence_relationships.head(15))

### ผล Pattern 2

พบ 142 คู่หน่วยงานย่อย–ผู้รับจ้าง รวม 2,421 โครงการ

## 4. Pattern 3 — การพึ่งพาผู้รับจ้างภายในจังหวัด

ใช้ส่วนแบ่งจำนวนโครงการและมูลค่าภายในจังหวัด เพื่อดูว่าความสัมพันธ์ที่พบในระดับหน่วยงานย่อยขยายถึงระดับจังหวัดหรือไม่

In [ ]:
valid_geographic_data = study_supplier_data.loc[
    study_supplier_data[province_column].notna()
    & ~study_supplier_data[province_column]
    .astype('string')
    .str.strip()
    .isin(['', '-', 'ไม่ระบุ'])
].copy()

province_supplier_relationships = (
    valid_geographic_data
    .groupby(
        [province_column, supplier_id_column],
        dropna=False
    )
    .agg(
        supplier_name=('supplier_name', 'first'),
        supplier_project_count=(project_id_column, 'nunique'),
        supplier_awarded_value=('supplier_awarded_value', 'sum')
    )
    .reset_index()
)

province_totals = (
    study_project_data.loc[
        study_project_data[province_column].notna()
        & ~study_project_data[province_column]
        .astype('string')
        .str.strip()
        .isin(['', '-', 'ไม่ระบุ'])
    ]
    .groupby(province_column, dropna=False)
    .agg(
        province_project_count=(project_id_column, 'nunique'),
        province_awarded_value=(awarded_price_column, 'sum')
    )
    .reset_index()
)

province_supplier_relationships = (
    province_supplier_relationships
    .merge(
        province_totals,
        on=province_column,
        how='left',
        validate='many_to_one'
    )
)

province_supplier_relationships['project_share_pct'] = (
    province_supplier_relationships['supplier_project_count']
    / province_supplier_relationships['province_project_count']
    * 100
)

province_supplier_relationships['value_share_pct'] = (
    province_supplier_relationships['supplier_awarded_value']
    / province_supplier_relationships['province_awarded_value']
    * 100
)

display(
    province_supplier_relationships
    .sort_values('project_share_pct', ascending=False)
    .head(15)
)

In [ ]:
province_minimum_options = [20, 50, 100]
supplier_minimum_options = [5, 10, 20]
share_threshold_options = [50, 75, 90]

geographic_records = []

for province_minimum in province_minimum_options:
    for supplier_minimum in supplier_minimum_options:
        for share_threshold in share_threshold_options:
            selected = province_supplier_relationships.loc[
                province_supplier_relationships['province_project_count'].ge(province_minimum)
                & province_supplier_relationships['supplier_project_count'].ge(supplier_minimum)
                & province_supplier_relationships['project_share_pct'].ge(share_threshold)
                & province_supplier_relationships['value_share_pct'].ge(share_threshold)
            ]

            geographic_records.append({
                'โครงการขั้นต่ำในจังหวัด': province_minimum,
                'โครงการขั้นต่ำของผู้รับจ้าง': supplier_minimum,
                'ส่วนแบ่งขั้นต่ำ (%)': share_threshold,
                'จำนวนคู่จังหวัด–ผู้รับจ้าง': len(selected)
            })

geographic_sensitivity = pd.DataFrame(geographic_records)
display(geographic_sensitivity)

eligible_province_relationships = (
    province_supplier_relationships.loc[
        province_supplier_relationships['province_project_count'].ge(20)
        & province_supplier_relationships['supplier_project_count'].ge(5)
    ]
)

maximum_geographic_share = pd.Series({
    'ส่วนแบ่งจำนวนโครงการสูงสุด (%)': (
        eligible_province_relationships['project_share_pct'].max()
    ),
    'ส่วนแบ่งมูลค่าสูงสุด (%)': (
        eligible_province_relationships['value_share_pct'].max()
    )
}, name='value')

display(maximum_geographic_share.to_frame())

### ผล Pattern 3

ไม่พบคู่จังหวัด–ผู้รับจ้างที่ครองทั้งจำนวนโครงการและมูลค่าตั้งแต่ 50% ขึ้นไปภายใต้เกณฑ์ที่ทดสอบ

## 5. การซ้อนทับ Pattern 1 และ Pattern 2

In [ ]:
study_review_data = project_data.loc[
    project_data['in_study_scope']
].copy()

study_review_data['pattern_1_only'] = (
    study_review_data['flag_pattern_1']
    & ~study_review_data['flag_pattern_2']
)

study_review_data['pattern_2_only'] = (
    ~study_review_data['flag_pattern_1']
    & study_review_data['flag_pattern_2']
)

study_review_data['pattern_1_and_2'] = (
    study_review_data['flag_pattern_1']
    & study_review_data['flag_pattern_2']
)

study_review_data['review_priority'] = np.select(
    [
        study_review_data['pattern_1_and_2'],
        study_review_data['flag_pattern_1']
        | study_review_data['flag_pattern_2']
    ],
    [
        'ตรวจสอบลำดับแรก',
        'พบ Pattern เดียว'
    ],
    default='ไม่พบ Pattern'
)

overlap_summary = pd.Series({
    'Pattern 1': study_review_data['flag_pattern_1'].sum(),
    'Pattern 2': study_review_data['flag_pattern_2'].sum(),
    'Pattern 1 และ Pattern 2': study_review_data['pattern_1_and_2'].sum(),
    'พบอย่างน้อยหนึ่ง Pattern': (
        study_review_data['flag_pattern_1']
        | study_review_data['flag_pattern_2']
    ).sum(),
    'ไม่พบ Pattern': (
        ~study_review_data['flag_pattern_1']
        & ~study_review_data['flag_pattern_2']
    ).sum()
}, name='project_count')

display(overlap_summary.to_frame())

In [ ]:
plot_data = pd.Series({
    'Pattern 1': study_review_data['flag_pattern_1'].sum(),
    'Pattern 2': study_review_data['flag_pattern_2'].sum(),
    'พบทั้ง Pattern 1 และ 2': study_review_data['pattern_1_and_2'].sum()
})

fig, ax = plt.subplots(figsize=(9, 4.5))

bars = ax.barh(
    plot_data.index,
    plot_data.values,
    color=[BLUE, BLUE, ORANGE],
    height=0.58
)

ax.bar_label(
    bars,
    labels=[f'{value:,.0f}' for value in plot_data.values],
    padding=5,
    fontsize=11,
    color=TEXT
)

ax.invert_yaxis()
ax.set_xlim(0, plot_data.max() * 1.18)
ax.set_title(
    'จำนวนโครงการตาม Pattern',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.tight_layout()

png_path = figure_dir / 'fig04_01_pattern_overlap.png'
svg_path = figure_dir / 'fig04_01_pattern_overlap.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')

In [ ]:
priority_project_ids = set(
    study_review_data.loc[
        study_review_data['pattern_1_and_2'],
        project_id_column
    ]
)

priority_projects = (
    study_review_data.loc[
        study_review_data[project_id_column].isin(priority_project_ids),
        [
            project_id_column,
            project_name_column,
            agency_column,
            subagency_column,
            province_column,
            method_column,
            budget_column,
            awarded_price_column,
            transaction_date_column,
            'flag_pattern_1',
            'flag_pattern_2',
            'review_priority'
        ]
    ]
    .sort_values(
        [subagency_column, transaction_date_column, budget_column],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

display(priority_projects.head(20))
print(f'โครงการตรวจสอบลำดับแรก: {len(priority_projects):,}')

### ผลการซ้อนทับ

โครงการที่พบทั้ง Pattern 1 และ Pattern 2 เป็นกลุ่มตรวจสอบลำดับแรก

## 6. สรุป

- Pattern 1 พบ 1,245 กลุ่ม รวม 5,623 โครงการ
- Pattern 2 พบ 142 คู่หน่วยงานย่อย–ผู้รับจ้าง รวม 2,421 โครงการ
- Pattern 3 ไม่พบคู่จังหวัด–ผู้รับจ้างที่ผ่านเกณฑ์ส่วนแบ่งขั้นต่ำ 50%
- กลุ่มตรวจสอบลำดับแรกคือโครงการที่พบทั้ง Pattern 1 และ Pattern 2
- ผลลัพธ์เป็นสัญญาณสำหรับเปิดเอกสารตรวจสอบเพิ่มเติม

## 7. บันทึกผลลัพธ์

In [ ]:
project_flags = project_data[
    [
        project_id_column,
        'in_study_scope',
        'flag_pattern_1',
        'flag_pattern_2'
    ]
].copy()

project_flags = project_flags.merge(
    study_review_data[
        [project_id_column, 'review_priority']
    ],
    on=project_id_column,
    how='left',
    validate='one_to_one'
)

output_objects = {
    'project_review_indicators_2569.csv': project_flags,
    'priority_review_projects_2569.csv': priority_projects,
    'repeated_near_500k_clusters_2569.csv': repeated_clusters,
    'high_supplier_dependence_2569.csv': high_dependence_relationships,
    'near_500k_sensitivity_2569.csv': near_ceiling_sensitivity,
    'cluster_size_sensitivity_2569.csv': cluster_size_sensitivity,
    'supplier_dependence_sensitivity_2569.csv': supplier_dependence_sensitivity,
    'province_supplier_summary_2569.csv': province_supplier_relationships,
    'geographic_supplier_dominance_sensitivity_2569.csv': geographic_sensitivity
}

export_records = []

for file_name, output_data in output_objects.items():
    output_path = processed_dir / file_name
    output_data.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig'
    )
    export_records.append({
        'file_name': file_name,
        'rows': len(output_data),
        'file_size_mb': output_path.stat().st_size / 1024**2
    })

export_summary = pd.DataFrame(export_records)
display(export_summary)